In [ ]:
"""Entrenamiento Offline RL del Otter con CQL — Colab T4.

Carga el dataset HDF5 generado por `agent/observe.py` (formato por episodios con
telemetría completa + log de acciones), construye el MDP (s, a, r, s', done) y
entrena un agente CQL con d3rlpy.

USO EN COLAB:
1. Subir `dataset_v1.h5` a Drive: `MyDrive/wakuseibokan/dataset_v1.h5`
2. File → Upload notebook → train_otter_cql.ipynb
3. Runtime → Change runtime type → GPU T4
4. Ejecutar las celdas en orden (Shift+Enter) o Runtime → Run all
"""

In [ ]:
# !pip install -q d3rlpy h5py
import os
import numpy as np
import h5py
import torch
from pathlib import Path
from typing import Dict, List, Tuple

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu:0"

In [ ]:
# En Colab descomentá las 4 líneas siguientes:
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = "/content/drive/MyDrive/wakuseibokan/dataset_v1.h5"
# OUTPUT_DIR  = "/content/drive/MyDrive/wakuseibokan/models/"

# Para correr local (smoke test):
DATASET_PATH = "data/dataset_v1.h5"
OUTPUT_DIR = "models/"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# Carga, encoders, reward y construcción de MDP — todo junto para no preocuparte
# por el orden de ejecución.

POS_SCALE = 2000.0
OBS_DIM = 12
ACT_DIM = 6


def load_episodes(path: str) -> Tuple[List[Dict], Dict]:
    """Carga episodios desde HDF5 con telemetría completa + log de acciones."""
    episodes = []
    with h5py.File(path, "r") as f:
        for name in sorted(f.keys()):
            g = f[name]
            ep = {k: g[k][:] for k in g.keys()}
            ep["_attrs"] = dict(g.attrs)
            episodes.append(ep)
        meta = dict(f.attrs)
    return episodes, meta


def encode_state(my_idx: int, other_idx: int, ep: Dict, t: int) -> np.ndarray:
    """12 features: mi pos norm + cos/sin azimuth + health + power +
    delta pos al enemigo + distancia + cos/sin bearing relativo + health enemigo."""
    pos_me  = ep["pos"][t, my_idx]
    pos_oth = ep["pos"][t, other_idx]
    az_me   = ep["azimuth"][t, my_idx]
    h_me    = ep["health"][t, my_idx]
    p_me    = ep["power"][t, my_idx]
    h_oth   = ep["health"][t, other_idx]

    dx = pos_oth[0] - pos_me[0]
    dz = pos_oth[2] - pos_me[2]
    dist = np.sqrt(dx * dx + dz * dz)
    bearing_world = np.arctan2(dz, dx)
    bearing_rel = bearing_world - az_me * np.pi / 180.0

    return np.array([
        pos_me[0] / POS_SCALE,
        pos_me[2] / POS_SCALE,
        np.cos(az_me * np.pi / 180.0),
        np.sin(az_me * np.pi / 180.0),
        np.clip(h_me / 1000.0, -1.0, 1.5),
        np.clip(p_me / 1000.0, 0.0, 1.5),
        dx / POS_SCALE,
        dz / POS_SCALE,
        np.clip(dist / POS_SCALE, 0.0, 3.0),
        np.cos(bearing_rel),
        np.sin(bearing_rel),
        np.clip(h_oth / 1000.0, -1.0, 1.5),
    ], dtype=np.float32)


def encode_action(ep: Dict, t: int) -> np.ndarray:
    """6 dims continuas en [-1, 1]: thrust, steering, turret_decl, cos+sin bearing, fire."""
    tb_rad = ep["act_turret_bearing"][t] * np.pi / 180.0
    return np.array([
        np.clip(ep["act_thrust"][t] / 10.0, -1.0, 1.0),
        np.clip(ep["act_steering"][t], -1.0, 1.0),
        np.clip(ep["act_turret_decl"][t] / 0.4, -1.0, 1.0),
        np.cos(tb_rad),
        np.sin(tb_rad),
        1.0 if ep["act_fire"][t] else -1.0,
    ], dtype=np.float32)


def compute_rewards_and_terminals(my_idx: int, other_idx: int,
                                   ep: Dict) -> Tuple[np.ndarray, np.ndarray]:
    """Reward shaping ESCALADO para CQL.

    Magnitudes chicas (|r| ≤ 5) para que la Q-function no explote. Si después
    escalas el modelo o usás más data, podés volver a magnitudes más grandes.
    """
    h_me  = ep["health"][:, my_idx].astype(np.float32)
    h_oth = ep["health"][:, other_idx].astype(np.float32)
    fire  = ep["act_fire"].astype(bool)
    n = len(h_me)
    rewards = np.zeros(n, dtype=np.float32)
    terminals = np.zeros(n, dtype=bool)

    for t in range(n):
        r = -0.001   # step cost
        if t > 0:
            dmg_dado = max(0.0, h_oth[t - 1] - h_oth[t])
            dmg_recib = max(0.0, h_me[t - 1] - h_me[t])
            r += 0.001 * dmg_dado - 0.001 * dmg_recib
        if t < len(fire) and fire[t]:
            r -= 0.0005
        if h_oth[t] <= 0 and (t == 0 or h_oth[t - 1] > 0):
            r += 5.0   # kill bonus
            terminals[t] = True
        if h_me[t] <= 0 and (t == 0 or h_me[t - 1] > 0):
            r -= 5.0   # death penalty
            terminals[t] = True
        rewards[t] = r
    terminals[-1] = True
    return rewards, terminals


def build_mdp_arrays(episodes: List[Dict], controlled_vid: int = 1):
    """Aplana episodios a (obs, actions, rewards, terminals)."""
    all_obs, all_act, all_rew, all_term = [], [], [], []

    for ep in episodes:
        vids = list(ep["vehicle_ids"])
        if controlled_vid not in vids:
            continue
        my_idx = vids.index(controlled_vid)
        if len(vids) == 2:
            other_idx = 1 - my_idx
        else:
            others = [v for v in vids if v != controlled_vid]
            other_idx = vids.index(others[0])

        n = ep["pos"].shape[0]
        n_act = len(ep["act_thrust"])
        usable = min(n, n_act)
        if usable < 2:
            continue

        rewards, terminals = compute_rewards_and_terminals(my_idx, other_idx, ep)

        for t in range(usable):
            all_obs.append(encode_state(my_idx, other_idx, ep, t))
            all_act.append(encode_action(ep, t))
            all_rew.append(rewards[t])
            all_term.append(terminals[t])

    obs_arr  = np.stack(all_obs).astype(np.float32)
    act_arr  = np.stack(all_act).astype(np.float32)
    rew_arr  = np.array(all_rew, dtype=np.float32)
    term_arr = np.array(all_term, dtype=bool)
    return obs_arr, act_arr, rew_arr, term_arr


print("✓ Helpers definidos: load_episodes, encode_state, encode_action, "
      "compute_rewards_and_terminals, build_mdp_arrays")

In [ ]:
episodes, meta = load_episodes(DATASET_PATH)
print(f"Cargados {len(episodes)} episodios. Meta: {meta}")
for i, ep in enumerate(episodes[:3]):
    print(f"  Ep {i}: {ep['_attrs'].get('n_ticks', '?')} ticks, "
          f"vehicles={list(ep['vehicle_ids'])}, "
          f"dist_fire={ep['_attrs'].get('params_dist_fire', '?'):.0f}m, "
          f"final_health={ep['health'][-1]}")

obs, actions, rewards, terminals = build_mdp_arrays(episodes, controlled_vid=1)
print(f"\nTransiciones totales: {len(obs)}")
print(f"  obs: {obs.shape}  actions: {actions.shape}")
print(f"  rewards: mean={rewards.mean():.3f} min={rewards.min():.1f} max={rewards.max():.1f}")
print(f"  terminals: {terminals.sum()}")

from d3rlpy.dataset import MDPDataset
mdp = MDPDataset(observations=obs, actions=actions, rewards=rewards, terminals=terminals)
print(f"MDPDataset construido: {len(mdp.episodes)} episodios")

In [ ]:
from d3rlpy.algos import CQLConfig

cql_config = CQLConfig(
    actor_learning_rate=1e-4,         # bajado de 3e-4 para estabilidad
    critic_learning_rate=1e-4,
    temp_learning_rate=1e-4,
    batch_size=256,
    gamma=0.95,                       # bajado de 0.99 → menos peso al futuro lejano
    tau=0.005,
    n_critics=2,
    initial_temperature=1.0,
    initial_alpha=1.0,
    alpha_threshold=10.0,
    conservative_weight=1.0,          # bajado de 5.0 → CQL menos agresivo
    n_action_samples=10,
)
cql = cql_config.create(device=DEVICE)

# 50k pasos para smoke test con dataset chico. Subir a 200k–500k después.
N_STEPS = 50_000
N_STEPS_PER_EPOCH = 5_000

print(f"Entrenando CQL por {N_STEPS} pasos ({N_STEPS // N_STEPS_PER_EPOCH} épocas)...")
cql.fit(
    mdp,
    n_steps=N_STEPS,
    n_steps_per_epoch=N_STEPS_PER_EPOCH,
    save_interval=10,
    experiment_name="otter_cql_v1",
)

In [ ]:
out_d3 = os.path.join(OUTPUT_DIR, "otter_cql_v1.d3")
out_pt = os.path.join(OUTPUT_DIR, "otter_cql_v1.pt")
cql.save_model(out_d3)
print(f"✓ Modelo d3rlpy: {out_d3}")

torch.save({
    "policy_state_dict": cql.impl.policy.state_dict() if cql.impl else None,
    "obs_dim": OBS_DIM,
    "action_dim": ACT_DIM,
    "config": {k: v for k, v in cql_config.__dict__.items()
               if isinstance(v, (int, float, str, bool))},
}, out_pt)
print(f"✓ State dict PyTorch: {out_pt}")

# Descargar (Colab):
# from google.colab import files
# files.download(out_pt)